# Modelo de Lenguaje GRU pequeño

El siguiente notebook contiene cómo: cargar textos, tokenizar, construir ventanas de entrenamiento, entrenar un modelo `Embedding -> GRU -> Linear`, medir perplexity y generar muestras.

Este notebook queda configurado por defecto para `multi3_nlu_es_hotels`, el dominio de hoteles de Multi3-NLU++. Si quieres volver a una prueba ultrarrapida, cambia `DATASET_NAME` a `toy_sintetico`.


## 1. Instalacion y entorno

En Colab puedes activar GPU en `Runtime > Change runtime type > T4 GPU` o similar. El codigo tambien funciona en CPU para los corpus pequenos, solo tarda mas.


In [25]:
#@title Instalacion y entorno
!pip -q install datasets sentencepiece "sympy>=1.13.3,<1.14"

import csv
import glob
import json
import math
import random
import re
import subprocess
from collections import Counter
from pathlib import Path

# Workaround for occasional Colab/PyTorch/SymPy import-state mismatches.
# If this cell was already run before the install line, restart the runtime once.
import sympy.core.symbol

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))


device: cpu


## 2. Configuracion

Para reproducir literalmente la version minima del reporte, deja `TOKENIZER = 'word'` o usa `char` si quieres un corpus ultrapequeno. El modelo por defecto usa embeddings de 128 y GRU de 256, dentro del rango recomendado.


In [26]:
#@title Hiperparametros y dataset
DATASET_NAME = 'multi3_nlu_es_hotels' #@param ['multi3_nlu_es_hotels', 'multi3_nlu_es_banking', 'multi3_nlu_es_all', 'toy_sintetico', 'simple_tico19_es', 'massive_es', 'mtop_es', 'tico19_es']
TOKENIZER = 'word' #@param ['word', 'char']
TEXT_FIELD_SIMPLE_TICO = 'simplification' #@param ['simplification', 'original']

LOWERCASE = True #@param {type:'boolean'}
MAX_TEXTS = 20000 #@param {type:'integer'}
MIN_FREQ = 1 #@param {type:'integer'}

SEQ_LEN = 48 #@param {type:'integer'}
STRIDE = 8 #@param {type:'integer'}
BATCH_SIZE = 64 #@param {type:'integer'}
EPOCHS = 15 #@param {type:'integer'}

EMB_DIM = 128 #@param {type:'integer'}
HIDDEN_DIM = 256 #@param {type:'integer'}
NUM_LAYERS = 1 #@param {type:'integer'}
LR = 3e-3 #@param {type:'number'}
GRAD_CLIP = 1.0 #@param {type:'number'}

assert TOKENIZER in {'word', 'char'}
assert DATASET_NAME in {'multi3_nlu_es_hotels', 'multi3_nlu_es_banking', 'multi3_nlu_es_all', 'toy_sintetico', 'simple_tico19_es', 'massive_es', 'mtop_es', 'tico19_es'}


## 3. Carga de corpus

Las funciones siguientes devuelven una lista de strings. Asi el resto del notebook no depende del formato original del dataset.


In [27]:
#@title Loaders de datasets espanoles
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

def run_cmd(cmd):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

def ensure_git_repo(url, path):
    path = Path(path)
    if not path.exists():
        run_cmd(['git', 'clone', '--depth', '1', url, str(path)])
    return path

def clean_text_list(texts):
    cleaned = []
    for text in texts:
        if not isinstance(text, str):
            continue
        text = re.sub(r'\s+', ' ', text.strip())
        if text:
            cleaned.append(text)
    return cleaned

def iter_json_records(obj):
    if isinstance(obj, list):
        for item in obj:
            yield from iter_json_records(item)
    elif isinstance(obj, dict):
        known = {'text', 'utterance', 'utt', 'sentence', 'original', 'simplification'}
        if known.intersection(obj.keys()):
            yield obj
        else:
            for value in obj.values():
                if isinstance(value, (dict, list)):
                    yield from iter_json_records(value)

def choose_text(record, preferred_keys):
    for key in preferred_keys:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value

    spanish_keys = ['es', 'es-ES', 'es_ES', 'spa', 'spa_Latn', 'spanish']
    for value in record.values():
        if isinstance(value, dict):
            for key in spanish_keys:
                nested = value.get(key)
                if isinstance(nested, str) and nested.strip():
                    return nested

    for value in record.values():
        if isinstance(value, str) and len(value.strip()) > 1:
            return value
    return None

def dataset_to_texts(ds, preferred_keys):
    texts = []
    if hasattr(ds, 'keys'):
        split_names = [s for s in ['train', 'validation', 'dev', 'test'] if s in ds]
        split_names = split_names or list(ds.keys())
        for split in split_names:
            for record in ds[split]:
                text = choose_text(record, preferred_keys)
                if text:
                    texts.append(text)
    else:
        for record in ds:
            text = choose_text(record, preferred_keys)
            if text:
                texts.append(text)
    return clean_text_list(texts)

def load_toy_sintetico(n=1500):
    alarm_times = ['a las 6', 'a las 7', 'a las 8', 'a las 9']
    days = ['hoy', 'manana', 'el lunes', 'el martes']
    alarm_forms = [
        'pon una alarma {day} {time}',
        'despiertame {day} {time}',
        'activa una alarma {day} {time}',
    ]
    cities = ['Monterrey', 'Madrid', 'Bogota', 'CDMX']
    weather_forms = [
        'que tiempo hara en {city}',
        'dime el clima en {city}',
        'como estara el tiempo en {city}',
    ]
    hotel_forms = [
        'quiero revisar mi reserva del hotel',
        'cancela mi reserva del hotel para manana',
        'necesito cambiar la fecha de llegada',
    ]
    rows = []
    for _ in range(n):
        family = random.choice(['alarm', 'weather', 'hotel'])
        if family == 'alarm':
            rows.append(random.choice(alarm_forms).format(day=random.choice(days), time=random.choice(alarm_times)))
        elif family == 'weather':
            rows.append(random.choice(weather_forms).format(city=random.choice(cities)))
        else:
            rows.append(random.choice(hotel_forms))
    return rows

def multi3_path_matches_domain(path, domain):
    if domain == 'all':
        return True
    parts = str(path).lower().replace('\\', '/').split('/')
    aliases = {
        'hotels': {'hotel', 'hotels'},
        'banking': {'bank', 'banking'},
    }.get(domain, {domain})
    return bool(aliases.intersection(parts))

def multi3_record_matches_domain(record, domain):
    if domain == 'all':
        return True
    uid = str(record.get('uid', '')).lower()
    if domain == 'hotels':
        return uid.startswith(('hotel_', 'hotels_'))
    if domain == 'banking':
        return uid.startswith(('bank_', 'banking_'))
    return False

def multi3_hf_json_paths(domain):
    from huggingface_hub import hf_hub_download, list_repo_files

    repo_id = 'uoe-nlp/multi3-nlu'
    files = list_repo_files(repo_id, repo_type='dataset')
    spanish_files = [p for p in files if p.startswith('spanish/') and p.endswith('.json')]
    domain_files = [p for p in spanish_files if multi3_path_matches_domain(p, domain)]
    selected_files = domain_files if domain_files else spanish_files
    paths = []
    for repo_file in selected_files:
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=repo_file,
            repo_type='dataset',
            local_dir=DATA_DIR / 'multi3-nlu-hf',
        )
        paths.append((Path(local_path), repo_file))
    return paths

def multi3_git_json_paths(domain):
    repo = ensure_git_repo('https://huggingface.co/datasets/uoe-nlp/multi3-nlu', DATA_DIR / 'multi3-nlu')
    all_paths = sorted(repo.glob('spanish/**/*.json'))
    domain_paths = [p for p in all_paths if multi3_path_matches_domain(p.relative_to(repo), domain)]
    selected_paths = domain_paths if domain_paths else all_paths
    return [(p, p.relative_to(repo).as_posix()) for p in selected_paths]

def load_multi3_nlu_es(domain='all'):
    texts = []
    try:
        json_paths = multi3_hf_json_paths(domain)
    except Exception as exc:
        print('No pude listar/descargar con huggingface_hub; intento git clone.', repr(exc))
        json_paths = multi3_git_json_paths(domain)

    if not json_paths:
        raise FileNotFoundError(f'No encontre archivos JSON espanoles para Multi3-NLU++ dominio={domain}')

    for fp, repo_file in json_paths:
        with fp.open(encoding='utf-8') as f:
            data = json.load(f)
        path_matches = multi3_path_matches_domain(repo_file, domain)
        for record in iter_json_records(data):
            if domain != 'all' and not (path_matches or multi3_record_matches_domain(record, domain)):
                continue
            text = choose_text(record, ['text', 'utterance', 'sentence'])
            if text:
                texts.append(text)

    texts = clean_text_list(texts)
    if not texts:
        raise ValueError(f'No encontre textos para Multi3-NLU++ dominio={domain}. Prueba multi3_nlu_es_all para diagnosticar.')
    return texts

def load_simple_tico19_es():
    repo = ensure_git_repo('https://github.com/MMU-TDMLab/SimpleTICO19.git', DATA_DIR / 'SimpleTICO19')
    texts = []
    csv_paths = sorted((repo / 'dataset').glob('simpletico19.*.es.csv'))
    for fp in csv_paths:
        with fp.open(encoding='utf-8', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                text = row.get(TEXT_FIELD_SIMPLE_TICO) or row.get('simplification') or row.get('original')
                if text:
                    texts.append(text)
    return clean_text_list(texts)

def load_massive_es():
    ds = load_dataset('AmazonScience/massive', 'es-ES')
    return dataset_to_texts(ds, ['utt', 'text', 'utterance'])

def load_mtop_es():
    last_error = None
    for args in [('tasksource/mtop', 'es'), ('tasksource/mtop',)]:
        try:
            ds = load_dataset(*args)
            texts = dataset_to_texts(ds, ['text', 'utterance', 'query', 'question'])
            if texts:
                return texts
        except Exception as exc:
            last_error = exc
    raise RuntimeError('No pude cargar MTOP desde Hugging Face') from last_error

def load_tico19_es():
    ds = load_dataset('SEACrowd/tico_19', trust_remote_code=True)
    return dataset_to_texts(ds, ['text', 'sentence', 'es', 'spa', 'spanish'])

LOADERS = {
    'toy_sintetico': load_toy_sintetico,
    'multi3_nlu_es_hotels': lambda: load_multi3_nlu_es('hotels'),
    'multi3_nlu_es_banking': lambda: load_multi3_nlu_es('banking'),
    'multi3_nlu_es_all': lambda: load_multi3_nlu_es('all'),
    'simple_tico19_es': load_simple_tico19_es,
    'massive_es': load_massive_es,
    'mtop_es': load_mtop_es,
    'tico19_es': load_tico19_es,
}

texts = LOADERS[DATASET_NAME]()
texts = clean_text_list(texts)
random.shuffle(texts)
if MAX_TEXTS and MAX_TEXTS > 0:
    texts = texts[:MAX_TEXTS]

if len(texts) < 3:
    raise ValueError(f'El corpus cargado es demasiado pequeno: {len(texts)} textos')

print(f'dataset: {DATASET_NAME}')
print(f'textos cargados: {len(texts):,}')
print('\nEjemplos:')
for sample_text in texts[:5]:
    print('-', sample_text[:220])


dataset: multi3_nlu_es_hotels
textos cargados: 1,009

Ejemplos:
- reserva una mesa para 5
- ¿a qué hora abre el gimnasio en la mañana?
- hasta el viernes
- ¿Por qué el servicio de limpieza no limpió mi habitación esta mañana?
- del 26 al 29 de mayo


## 4. Normalizacion, tokenizacion y vocabulario

Para corpus diminutos puedes usar `char`. Para Multi3-NLU++, Simple TICO-19 o MASSIVE, `word` suele ser suficiente para aprender la canalizacion completa antes de pasar a BPE.


In [28]:
#@title Tokenizacion y vocabulario
WORD_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def normalize(text):
    text = re.sub(r'\s+', ' ', text.strip())
    return text.lower() if LOWERCASE else text

def tokenize(text):
    text = normalize(text)
    if TOKENIZER == 'char':
        return list(text)
    return WORD_RE.findall(text)

split_at = max(1, int(0.9 * len(texts)))
train_texts = texts[:split_at]
val_texts = texts[split_at:] or texts[:max(1, min(100, len(texts)))]

counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))

SPECIAL = ['<pad>', '<unk>', '<bos>', '<eos>']
itos = SPECIAL + [tok for tok, count in counter.most_common() if count >= MIN_FREQ and tok not in SPECIAL]
stoi = {tok: idx for idx, tok in enumerate(itos)}

PAD = stoi['<pad>']
UNK = stoi['<unk>']
BOS = stoi['<bos>']
EOS = stoi['<eos>']

def encode_texts(text_list):
    ids = []
    for text in text_list:
        ids.append(BOS)
        ids.extend(stoi.get(tok, UNK) for tok in tokenize(text))
        ids.append(EOS)
    return torch.tensor(ids, dtype=torch.long)

train_ids = encode_texts(train_texts)
val_ids = encode_texts(val_texts)
if len(val_ids) <= SEQ_LEN + 1:
    val_ids = train_ids.clone()

print(f'train_texts: {len(train_texts):,}')
print(f'val_texts: {len(val_texts):,}')
print(f'vocab_size: {len(itos):,}')
print(f'train_tokens: {len(train_ids):,}')
print(f'val_tokens: {len(val_ids):,}')


train_texts: 908
val_texts: 101
vocab_size: 1,086
train_tokens: 10,386
val_tokens: 1,173


## 5. Dataset de ventanas causales

Cada ejemplo usa `x = tokens[t:t+SEQ_LEN]` y `y = tokens[t+1:t+SEQ_LEN+1]`. Esa es la tarea clasica de predecir el siguiente token.


In [29]:
#@title Ventanas de entrenamiento
class WindowDataset(Dataset):
    def __init__(self, ids, seq_len=32, stride=1):
        if len(ids) <= seq_len + 1:
            raise ValueError('El corpus codificado es mas corto que SEQ_LEN. Reduce SEQ_LEN o carga mas texto.')
        self.ids = ids
        self.seq_len = seq_len
        self.stride = max(1, stride)
        self.n = 1 + (len(ids) - seq_len - 1) // self.stride

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        start = idx * self.stride
        x = self.ids[start : start + self.seq_len]
        y = self.ids[start + 1 : start + self.seq_len + 1]
        return x, y

train_ds = WindowDataset(train_ids, seq_len=SEQ_LEN, stride=STRIDE)
val_ds = WindowDataset(val_ids, seq_len=SEQ_LEN, stride=STRIDE)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'train_windows: {len(train_ds):,}')
print(f'val_windows: {len(val_ds):,}')


train_windows: 1,293
val_windows: 141


## 6. Modelo minimo: Embedding -> GRU -> Linear

Esta es la arquitectura recomendada en el reporte para empezar: pequena, facil de leer, y suficiente para aprender el flujo completo de un language model causal.


In [30]:
#@title Modelo GRU minimo
class TinyGRULM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, num_layers=1):
        super().__init__()
        dropout = 0.1 if num_layers > 1 else 0.0
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        x = self.emb(x)
        out, h = self.rnn(x, h)
        logits = self.head(out)
        return logits, h

model = TinyGRULM(
    vocab_size=len(itos),
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'parametros: {n_params:,}')


TinyGRULM(
  (emb): Embedding(1086, 128, padding_idx=0)
  (rnn): GRU(128, 256, batch_first=True)
  (head): Linear(in_features=256, out_features=1086, bias=True)
)
parametros: 714,558


## 7. Entrenamiento y perplexity

La perplexity deberia bajar rapido en `toy_sintetico`. En corpus reales puede bajar mas lento, pero aun asi deberias ver una tendencia clara si el dataset cargo bien.


In [31]:
#@title Loop de entrenamiento
def safe_ppl(avg_loss):
    return float('inf') if avg_loss > 20 else math.exp(avg_loss)

def run_epoch(loader, train=True):
    model.train(mode=train)
    total_loss = 0.0
    total_tokens = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits, _ = model(x)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()

            n_tokens = (y != PAD).sum().item()
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens

    avg_loss = total_loss / max(1, total_tokens)
    return avg_loss, safe_ppl(avg_loss)

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_ppl = run_epoch(train_dl, train=True)
    val_loss, val_ppl = run_epoch(val_dl, train=False)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'train_ppl': train_ppl, 'val_ppl': val_ppl})
    print(
        f'epoch={epoch:02d} '
        f'train_loss={train_loss:.4f} train_ppl={train_ppl:.2f} '
        f'val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}'
    )


epoch=01 train_loss=4.8874 train_ppl=132.60 val_loss=4.1657 val_ppl=64.44
epoch=02 train_loss=3.3352 train_ppl=28.08 val_loss=3.7283 val_ppl=41.61
epoch=03 train_loss=2.6760 train_ppl=14.53 val_loss=3.5889 val_ppl=36.20
epoch=04 train_loss=2.2306 train_ppl=9.31 val_loss=3.5418 val_ppl=34.53
epoch=05 train_loss=1.8731 train_ppl=6.51 val_loss=3.5389 val_ppl=34.43
epoch=06 train_loss=1.5809 train_ppl=4.86 val_loss=3.5952 val_ppl=36.42
epoch=07 train_loss=1.3302 train_ppl=3.78 val_loss=3.6508 val_ppl=38.50
epoch=08 train_loss=1.1257 train_ppl=3.08 val_loss=3.7533 val_ppl=42.66
epoch=09 train_loss=0.9497 train_ppl=2.59 val_loss=3.8433 val_ppl=46.68
epoch=10 train_loss=0.8034 train_ppl=2.23 val_loss=3.9418 val_ppl=51.51
epoch=11 train_loss=0.6698 train_ppl=1.95 val_loss=4.0693 val_ppl=58.52
epoch=12 train_loss=0.5492 train_ppl=1.73 val_loss=4.1885 val_ppl=65.92
epoch=13 train_loss=0.4444 train_ppl=1.56 val_loss=4.2893 val_ppl=72.92
epoch=14 train_loss=0.3542 train_ppl=1.43 val_loss=4.3839 va

## 8. Generacion

La generacion aqui usa muestreo multinomial con temperatura y `top_k`. Para corpus pequenos, prueba temperaturas entre `0.7` y `1.0`.


In [32]:
#@title Muestreo de texto
@torch.no_grad()
def sample(prompt='quiero revisar', max_new=40, temperature=0.9, top_k=30):
    model.eval()
    token_ids = [BOS] + [stoi.get(tok, UNK) for tok in tokenize(prompt)]

    for _ in range(max_new):
        context = torch.tensor([token_ids[-SEQ_LEN:]], dtype=torch.long, device=device)
        logits, _ = model(context)
        logits = logits[:, -1, :] / max(temperature, 1e-6)

        if top_k and top_k > 0 and top_k < logits.size(-1):
            values, _ = torch.topk(logits, top_k)
            cutoff = values[:, -1].unsqueeze(-1)
            logits = torch.where(logits < cutoff, torch.full_like(logits, -float('inf')), logits)

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        if next_id == EOS:
            break
        token_ids.append(next_id)

    out_tokens = [itos[i] for i in token_ids[1:] if i not in {PAD, EOS}]
    if TOKENIZER == 'char':
        return ''.join(tok for tok in out_tokens if tok not in {'<bos>', '<unk>'})

    text = ' '.join(tok for tok in out_tokens if tok != '<bos>')
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    return text

TOY_PROMPTS = ['quiero', 'pon una alarma', 'los sintomas', 'reserva del hotel']
HOTEL_PROMPTS = ['quiero reservar', 'necesito cambiar', 'reserva del hotel', 'habitación para']
DEFAULT_PROMPTS = ['quiero', 'necesito', 'dime', 'puedo']

if DATASET_NAME == 'toy_sintetico':
    prompts = TOY_PROMPTS
elif DATASET_NAME == 'multi3_nlu_es_hotels':
    prompts = HOTEL_PROMPTS
else:
    prompts = DEFAULT_PROMPTS

for prompt in prompts:
    print(f'PROMPT: {prompt}')
    print(sample(prompt=prompt, max_new=40, temperature=0.9, top_k=30))
    print()


PROMPT: quiero reservar
quiero reservar un hotel para dos adultos y un chico

PROMPT: necesito cambiar
necesito cambiar mi reserva

PROMPT: reserva del hotel
reserva del hotel

PROMPT: habitación para
habitación para 3 adultos



## 9. Guardar checkpoint

El checkpoint guarda pesos, vocabulario y configuracion basica. En Colab puedes descargar el archivo desde el panel lateral de archivos.


In [33]:
#@title Guardar modelo
SAVE_PATH = 'tiny_gru_lm_es.pt'
checkpoint = {
    'model_state_dict': model.state_dict(),
    'itos': itos,
    'stoi': stoi,
    'config': {
        'dataset_name': DATASET_NAME,
        'tokenizer': TOKENIZER,
        'lowercase': LOWERCASE,
        'seq_len': SEQ_LEN,
        'emb_dim': EMB_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
    },
    'history': history,
}
torch.save(checkpoint, SAVE_PATH)
print(f'checkpoint guardado en: {SAVE_PATH}')


checkpoint guardado en: tiny_gru_lm_es.pt


## Siguientes experimentos utiles

- Usa `multi3_nlu_es_hotels` para entrenar solo con hoteles, o `multi3_nlu_es_banking` para banca.
- Prueba `TOKENIZER = 'char'` si `SEQ_LEN` pequeno o vocabulario grande te da problemas.
- Sube `EPOCHS` a 10-20 si la perplexity de validacion sigue bajando.
- Cuando esta version funcione, el siguiente paso natural es reemplazar `word/char` por BPE con `sentencepiece`.
